# Generate `gamma_mixed_final.npy`

Run this notebook first when you want to rebuild the mean-field density from the raw `fcidump_cycle_6` and `dets.npz` files.

It starts from the reference determinant, repeatedly solves the overlapping MFA fragments, updates the orbital occupations, and saves the resulting density as `Outputs/meanfield_active/outs_extraction_autodets/gamma_mixed_final.npy`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# Locate the FRASCI folder whether Jupyter starts here or one level up.
CWD = Path.cwd().resolve()
FLOW_ROOT = None
for base in [CWD] + list(CWD.parents):
    if (base / "FRASCI" / "__init__.py").exists() and (base / "data" / "fcidump_cycle_6").exists():
        FLOW_ROOT = base
        break
    nested = base / "FRASCI"
    if (nested / "FRASCI" / "__init__.py").exists() and (nested / "data" / "fcidump_cycle_6").exists():
        FLOW_ROOT = nested
        break
if FLOW_ROOT is None:
    raise RuntimeError("Could not locate the FRASCI project folder")

if str(FLOW_ROOT) not in sys.path:
    sys.path.insert(0, str(FLOW_ROOT))

from FRASCI.mfa import bootstrap_gamma

FCIDUMP = FLOW_ROOT / "data" / "fcidump_cycle_6"
DETS_NPZ = FLOW_ROOT / "data" / "dets.npz"
GAMMA_DIR = FLOW_ROOT / "Outputs" / "meanfield_active" / "outs_extraction_autodets"
GAMMA_PATH = GAMMA_DIR / "gamma_mixed_final.npy"
METADATA_PATH = GAMMA_DIR / "gamma_mixed_final_metadata.json"

print(f"FLOW_ROOT     = {FLOW_ROOT}")
print(f"FCIDUMP       = {FCIDUMP}")
print(f"DETS_NPZ      = {DETS_NPZ}")
print(f"GAMMA_PATH    = {GAMMA_PATH}")

for required in (FCIDUMP, DETS_NPZ):
    if not required.exists():
        raise FileNotFoundError(required)


In [ ]:
# These are the same light selected-CI settings used by the D1/D2 notebook.
# Increase MAX_ITER if you want a tighter self-consistent gamma.
TRIMCI_CONFIG = {
    "threshold": 0.06,
    "max_final_dets": "auto",
    "max_rounds": 2,
    "num_runs": 1,
    "pool_build_strategy": "heat_bath",
    "verbose": False,
}

MAX_ITER = 12
TOL = 1.0e-4
MIXING = 0.30

display(Markdown("### Gamma extraction settings"))
print("TRIMCI_CONFIG:", TRIMCI_CONFIG)
print("MAX_ITER:", MAX_ITER)
print("TOL:", TOL)
print("MIXING:", MIXING)


In [ ]:
gamma = bootstrap_gamma(
    fcidump_path=str(FCIDUMP),
    dets_npz_path=str(DETS_NPZ),
    config=TRIMCI_CONFIG,
    n_iter=MAX_ITER,
    tol=TOL,
    mixing=MIXING,
    output_path=str(GAMMA_PATH),
    metadata_path=str(METADATA_PATH),
)


In [ ]:
loaded = np.load(GAMMA_PATH)

print("Saved gamma diagnostics")
print("=" * 48)
print(f"path        : {GAMMA_PATH}")
print(f"shape       : {loaded.shape}")
print(f"sum         : {loaded.sum():.8f}")
print(f"min / max   : {loaded.min():.8f} / {loaded.max():.8f}")
print(f"metadata    : {METADATA_PATH}")

if loaded.ndim != 1:
    raise ValueError(f"Expected a diagonal gamma vector, got shape {loaded.shape}")
if not np.isclose(loaded.sum(), 54.0, atol=1e-8):
    raise ValueError(f"Gamma should sum to 54 electrons, got {loaded.sum()}")

display(Markdown(f"""
### Ready

`gamma_mixed_final.npy` is now available at:

```text
{GAMMA_PATH}
```

You can now run `FRASCI_Results.ipynb`.
"""))
